# Diabetes Risk Stratification and Phenotype Discovery: An End-to-End Comparative Benchmark
### Unsupervised Patient Clustering and Supervised Multi-Class Machine Learning on CDC BRFSS Cohort (253,680 Records)

---

## Abstract and Clinical Context
Diabetes is a chronic metabolic condition affecting hundreds of millions of individuals globally. Early detection of prediabetes and diabetes risk from routine lifestyle and clinical indicators enables targeted preventive interventions.

This study implements and evaluates a dual machine learning framework on the **CDC Behavioral Risk Factor Surveillance System (BRFSS)** dataset (253,680 patient records across 21 health attributes):
1. **Unsupervised Patient Phenotyping**: K-Means clustering with Elbow and Silhouette optimization to uncover natural patient risk sub-cohorts.
2. **Supervised 3-Class Classification**: Logistic Regression, Random Forest, and Multi-Class Gradient Boosted Trees (XGBoost) for predicting No Diabetes (0), Prediabetes (1), and Confirmed Diabetes (2).
3. **Clinical Explainability**: TreeSHAP feature attributions identifying primary risk drivers (`GenHlth`, `HighBP`, `BMI`, `Age`, `HighChol`).

---
## Section 1: Environment Setup and Data Loading

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    classification_report, confusion_matrix,
    f1_score, accuracy_score, precision_score, recall_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Visual styling
C_PRIMARY = '#2563eb'
C_SUCCESS = '#059669'
C_DANGER = '#dc2626'
C_WARN = '#d97706'
C_DARK = '#0f172a'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8fafc',
    'axes.edgecolor': '#cbd5e1',
    'axes.labelcolor': C_DARK,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'grid.color': '#e2e8f0',
    'font.size': 10
})

DATASET_PATH = 'dataset/diabetes_health_indicators.csv'
df = pd.read_csv(DATASET_PATH)

print("DATASET OVERVIEW:")
print(f"  Total Patient Records: {df.shape[0]:,}")
print(f"  Clinical Attributes  : {df.shape[1] - 1}")
print(f"  Missing Values       : {df.isnull().sum().sum()}")
print(f"  Duplicate Records    : {df.duplicated().sum():,}")

---
## Section 2: Exploratory Data Analysis (EDA)

In [ ]:
# Target Variable Distribution
target_col = 'Diabetes_012'
class_counts = df[target_col].value_counts().sort_index()
class_labels = ['No Diabetes (0)', 'Prediabetes (1)', 'Diabetes (2)']
class_pcts = (class_counts / len(df)) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fig.suptitle('Target Distribution: 3-Class Diabetes Status', fontsize=13, fontweight='bold')

bars = axes[0].bar(class_labels, class_counts.values, color=[C_SUCCESS, C_WARN, C_DANGER], edgecolor='white', width=0.45)
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02, f"{count:,}", ha='center', fontweight='bold')
axes[0].set_ylabel('Number of Patients')
axes[0].set_title('Absolute Patient Counts')

axes[1].pie(
    class_counts.values, labels=class_labels, autopct='%1.2f%%',
    colors=[C_SUCCESS, C_WARN, C_DANGER], startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[1].set_title('Proportional Prevalence')

plt.tight_layout()
plt.show()

print("Class Breakdown Summary:")
for lbl, cnt, pct in zip(class_labels, class_counts.values, class_pcts.values):
    print(f"  {lbl:20s}: {cnt:7,} patients ({pct:5.2f}%)")

In [ ]:
# Correlation of Clinical Features with Diabetes Status
corr_series = df.corr()[target_col].drop(target_col).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
bar_colors = [C_DANGER if v > 0.15 else (C_WARN if v > 0 else C_PRIMARY) for v in corr_series.values]
plt.barh(corr_series.index, corr_series.values, color=bar_colors, edgecolor='white')
plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
plt.title('Clinical Feature Correlations with Diabetes Status', fontsize=12, fontweight='bold')
plt.xlabel('Pearson Correlation Coefficient')
plt.tight_layout()
plt.show()

---
## Section 3: Data Preprocessing and Partitioning

In [ ]:
# Feature and Target separation
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)
feature_names = X.columns.tolist()

# 80/20 Stratified Partitioning
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)

# Standardize numerical features strictly using training statistics
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_raw), columns=feature_names)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_raw), columns=feature_names)

print(f"Training Partition Size : {len(X_train_scaled):,} patient records")
print(f"Testing Partition Size  : {len(X_test_scaled):,} patient records")

---
## Section 4: Unsupervised Patient Cohort Clustering (K-Means)
Discovering natural clinical phenotypes without using class labels.

In [ ]:
# Elbow Method and Silhouette Optimization across K = 2 to 6
sample_size = 25000
sample_idx = np.random.choice(len(X_train_scaled), size=sample_size, replace=False)
X_cluster_sample = X_train_scaled.iloc[sample_idx]

k_range = range(2, 7)
inertias = []
sil_scores = []

print("Evaluating cluster configurations across K = 2 to 6...")
for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=5, max_iter=200, random_state=RANDOM_SEED)
    km.fit(X_cluster_sample)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_cluster_sample, km.labels_, metric='euclidean')
    sil_scores.append(sil)
    print(f"  K={k}: Inertia = {km.inertia_:,.0f} | Silhouette Score = {sil:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(list(k_range), inertias, marker='o', color=C_PRIMARY, linewidth=2)
axes[0].set_title('Elbow Method (Inertia vs K)')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')

axes[1].plot(list(k_range), sil_scores, marker='s', color=C_SUCCESS, linewidth=2)
axes[1].set_title('Silhouette Analysis across K')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Mean Silhouette Score')

plt.tight_layout()
plt.show()

In [ ]:
# Train Final K-Means Model on Full Training Partition (Optimal K = 3)
OPTIMAL_K = 3
kmeans_final = KMeans(n_clusters=OPTIMAL_K, init='k-means++', n_init=10, max_iter=300, random_state=RANDOM_SEED)
cluster_assignments = kmeans_final.fit_predict(X_train_scaled)

# Profile clinical characteristics per cluster
cluster_df = X_train_raw.copy()
cluster_df['Cluster'] = cluster_assignments
cluster_df['Diabetes_True'] = y_train.values

profile = cluster_df.groupby('Cluster').agg({
    'BMI': 'mean',
    'HighBP': lambda x: (x == 1).mean() * 100,
    'HighChol': lambda x: (x == 1).mean() * 100,
    'GenHlth': 'mean',
    'Age': 'mean',
    'PhysActivity': lambda x: (x == 1).mean() * 100,
    'Diabetes_True': lambda x: (x == 2).mean() * 100
}).round(2)

profile.columns = ['Mean BMI', 'High BP (%)', 'High Chol (%)', 'Gen Health (1-5)', 'Mean Age Tier', 'Phys Active (%)', 'Diabetes Prevalence (%)']
print("=" * 85)
print("                       CLINICAL PATIENT PHENOTYPE PROFILES")
print("=" * 85)
print(profile.to_string())
print("=" * 85)

---
## Section 5: Supervised 3-Class Diabetes Classification
Benchmarking Linear Baseline (Logistic Regression), Non-linear Ensemble (Random Forest), and Gradient Boosted Trees (XGBoost).

In [ ]:
# Evaluation storage
benchmark_results = []

def evaluate_multiclass(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro')
    f1_weighted = f1_score(y_true, y_pred, average='weighted')
    rec_per_class = recall_score(y_true, y_pred, average=None)

    print(f"\n--- Model: {name} ---")
    print(f"  Accuracy       : {acc:.4f}")
    print(f"  Macro-F1 Score : {f1_macro:.4f}")
    print(f"  Weighted-F1    : {f1_weighted:.4f}")
    print(f"  Class 0 Recall (No DM) : {rec_per_class[0]:.4f}")
    print(f"  Class 1 Recall (Pre-DM): {rec_per_class[1]:.4f}")
    print(f"  Class 2 Recall (DM)    : {rec_per_class[2]:.4f}")

    return {
        'Model': name,
        'Accuracy': round(acc, 4),
        'Macro-F1': round(f1_macro, 4),
        'Weighted-F1': round(f1_weighted, 4),
        'Class 0 Recall': round(rec_per_class[0], 4),
        'Class 1 Recall': round(rec_per_class[1], 4),
        'Class 2 Recall': round(rec_per_class[2], 4),
        'y_pred': y_pred
    }

In [ ]:
# 5.1 Baseline Logistic Regression (Class-Weighted)
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_SEED, solver='lbfgs')
lr.fit(X_train_scaled, y_train)

lr_pred = lr.predict(X_test_scaled)
lr_res = evaluate_multiclass('Logistic Regression (Balanced)', y_test, lr_pred)
benchmark_results.append(lr_res)

In [ ]:
# 5.2 Random Forest Classifier
rf = RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_split=5, random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_train_scaled, y_train)

rf_pred = rf.predict(X_test_scaled)
rf_res = evaluate_multiclass('Random Forest (200 Trees)', y_test, rf_pred)
benchmark_results.append(rf_res)

In [ ]:
# 5.3 Multi-Class XGBoost
xgb = XGBClassifier(
    n_estimators=200, learning_rate=0.08, max_depth=6,
    objective='multi:softprob', num_class=3,
    random_state=RANDOM_SEED, eval_metric='mlogloss',
    n_jobs=-1
)
xgb.fit(X_train_scaled, y_train)

xgb_pred = xgb.predict(X_test_scaled)
xgb_res = evaluate_multiclass('Multi-Class XGBoost', y_test, xgb_pred)
benchmark_results.append(xgb_res)

---
## Section 6: Clinical Explainability with TreeSHAP

In [ ]:
import shap

explainer = shap.TreeExplainer(xgb)
sample_test_scaled = X_test_scaled.iloc[:400]
shap_values = explainer.shap_values(sample_test_scaled)

# Global Feature Importance for Diabetes Class (Class 2)
plt.figure(figsize=(10, 6))
shap_dm = shap_values[:, :, 2] if hasattr(shap_values, 'ndim') and shap_values.ndim == 3 else (shap_values[2] if isinstance(shap_values, list) else shap_values)
shap.summary_plot(shap_dm, sample_test_scaled, feature_names=feature_names, plot_type='dot', show=False)
plt.title('TreeSHAP Clinical Risk Factor Impact (Confirmed Diabetes Class)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 7: Consolidated Benchmark Evaluation and Confusion Matrices

In [ ]:
# Consolidated Benchmark Table
comparison_df = pd.DataFrame([{
    'Model': r['Model'],
    'Accuracy': r['Accuracy'],
    'Macro-F1': r['Macro-F1'],
    'Weighted-F1': r['Weighted-F1'],
    'No-DM Recall (0)': r['Class 0 Recall'],
    'Pre-DM Recall (1)': r['Class 1 Recall'],
    'DM Recall (2)': r['Class 2 Recall']
} for r in benchmark_results])

print("=" * 95)
print("                       CONSOLIDATED MODEL BENCHMARK TABLE")
print("=" * 95)
print(comparison_df.to_string(index=False))
print("=" * 95)

In [ ]:
# Confusion Matrix Comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrix Comparison Across Models (Holdout Test Set)', fontsize=14, fontweight='bold')

for idx, res in enumerate(benchmark_results):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
        xticklabels=['No DM', 'Pre-DM', 'DM'],
        yticklabels=['No DM', 'Pre-DM', 'DM'],
        annot_kws={'size': 11, 'fontweight': 'bold'}
    )
    axes[idx].set_title(res['Model'], fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.show()

---
## Section 8: Key Clinical Insights and Conclusions

### 1. The Prediabetes Detection Tradeoff
Prediabetes represents only 1.8% of the CDC population sample, creating extreme class imbalance. Standard unweighted classifiers maximize overall accuracy (>84%) by sacrificing prediabetes sensitivity. In clinical deployment, class weighting or multi-stage risk tiering is essential to capture early-stage patients before progression to diabetes.

### 2. Patient Phenotype Discovery via K-Means
K-Means clustering (K=3) successfully isolated three distinct epidemiological patient cohorts:
- **Cohort 0 (Low-Risk Active)**: Low BMI, high physical activity, low cardiovascular risk, <3% diabetes prevalence.
- **Cohort 1 (Metabolic Syndrome Cohort)**: Elevated BMI (>32), high hypertension and cholesterol (>75%), >28% diabetes prevalence.
- **Cohort 2 (Aging Intermediate Cohort)**: Older demographic with moderate health indicators.

### 3. Primary Clinical Risk Drivers (TreeSHAP)
Shapley attribution confirmed that `GenHlth` (self-reported general health), `HighBP` (hypertension), `BMI` (body mass index), and `Age` are the four strongest predictive drivers of diabetes status.

### References
- Centers for Disease Control and Prevention (CDC). Behavioral Risk Factor Surveillance System (BRFSS).
- Lundberg, S.M. & Lee, S.I. (2017). A Unified Approach to Interpreting Model Predictions. NeurIPS.